In [ ]:
from pathlib import Path

In [11]:
import ffmpeg

for threshold in [0.3, 0.5, 0.7]:
    video_path = Path("../tests/videos/抖音2026528-272271.mp4")
    output_dir = Path(f"../tests/videos/frames/{threshold}")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_pattern = str((output_dir / "frame_%03d.png").resolve())
    print(output_pattern)
    try:
        (
            ffmpeg.input(str(video_path))
            .filter("select", "gt(scene,0.3)")
            .output(output_pattern, vsync="vfr")
            .run(overwrite_output=True, capture_stderr=True)
        )
    except ffmpeg.Error as e:
        print("Error:", e.stderr.decode())

D:\HKU\video_structure_transform\backend\tests\videos\frames\0.3\frame_%03d.png
D:\HKU\video_structure_transform\backend\tests\videos\frames\0.5\frame_%03d.png
D:\HKU\video_structure_transform\backend\tests\videos\frames\0.7\frame_%03d.png


In [ ]:
from scenedetect import ContentDetector, detect, open_video
from scenedetect.output import save_images

video = open_video(str(video_path))
scene_list = detect(str(video_path), ContentDetector())
save_images(scene_list, video=video, num_images=1)

{0: ['抖音2026528-272271-Scene-001-01.jpg'],
 1: ['抖音2026528-272271-Scene-002-01.jpg'],
 2: ['抖音2026528-272271-Scene-003-01.jpg'],
 3: ['抖音2026528-272271-Scene-004-01.jpg'],
 4: ['抖音2026528-272271-Scene-005-01.jpg'],
 5: ['抖音2026528-272271-Scene-006-01.jpg'],
 6: ['抖音2026528-272271-Scene-007-01.jpg'],
 7: ['抖音2026528-272271-Scene-008-01.jpg'],
 8: ['抖音2026528-272271-Scene-009-01.jpg'],
 9: ['抖音2026528-272271-Scene-010-01.jpg'],
 10: ['抖音2026528-272271-Scene-011-01.jpg'],
 11: ['抖音2026528-272271-Scene-012-01.jpg'],
 12: ['抖音2026528-272271-Scene-013-01.jpg'],
 13: ['抖音2026528-272271-Scene-014-01.jpg'],
 14: ['抖音2026528-272271-Scene-015-01.jpg'],
 15: ['抖音2026528-272271-Scene-016-01.jpg'],
 16: ['抖音2026528-272271-Scene-017-01.jpg'],
 17: ['抖音2026528-272271-Scene-018-01.jpg'],
 18: ['抖音2026528-272271-Scene-019-01.jpg']}

In [21]:
STRUCTURE_EXTRACTION_PROMPT = """
# Role
你是一个顶级的视频多媒体分析专家。你的任务是深度解构用户上传的视频，将其拆分为独立的 “Beat”（分镜/叙事单元），并提取出极其精准的视觉、听觉和剪辑风格参数，以便后续的 AI 视频生成 Agent 能够完美复刻该视频的风格。

## 什么是 Beat？
A beat is one discrete scene or segment of the video — a chunk of narration + visuals that covers a single idea (e.g. hook, story, proof, CTA).
Think of beats as the chapters of your video. A typical short video has 3–5 beats, each with its own mood, assets, and animations.

---

## 分析与提取任务
请仔细审视视频，并按以下维度对每一个 Beat 进行绝对量化的拆解：

### 1. 结构与时间轴 (Structure & Timeline)
- 识别每个 Beat 的精确起止时间戳（用于 `ffmpeg` 裁剪，精确到毫秒，如 `00:00:02.500`）。
- 提炼该 Beat 在核心叙事中的功能角色（例如：Hook 引子、Story 故事展开、Core Proof 核心论证、CTA 行动召唤）。

### 2. 视觉与资产风格 (Visuals & Asset Style)
- **画面构图 (Composition):** 主体位置、景别（特写/中景/全景）、视角（平视/俯视/仰视）。
- **色彩与光影 (Color & Lighting):** 主色调（Hex 码或色系）、饱和度、对比度、光照类型（明亮/暗调/赛博朋克霓虹等）。
- **资产类型 (Asset Type):** 画面主体是真实实拍、3D 渲染、2D 插画、高科技 UI 动效、还是纯文字排版？
- **背景风格 (Background):** 背景是实景、纯色、渐变还是动态粒子？

### 3. 动态与转场 (Motion & Transitions)
- **镜头运动 (Camera Movement):** 缓慢推近、平移、旋转还是静态？
- **主体动画 (Object Animation):** 画面元素的入场和出场动画（如：淡入、从底部弹入、淡出）。
- **转场特效 (Transitions):** 该 Beat 结束时使用了什么转场（如：硬切、闪白、擦除、无缝缩放）？

### 4. 文本与排版 (Typography & Layout)
- **字幕/花字样式 (Subtitle Style):** 字体类型（衬线/无衬线/手写）、颜色、描边、阴影。
- **排版位置 (Layout):** 文字在画面中的绝对/相对位置（如：居中、底部 1/4 处）。
- **文本动画 (Text Animation):** 逐字跳动、打字机效果、全显放大等。

### 5. 音频与节奏 (Audio & Pacing)
- **文案内容 (Narration):** 准确转录该 Beat 对应的语音文案（Narration Line）。
- **剪辑节奏 (Pacing):** 该 Beat 内的画面切换频率（如：快节奏 0.5s 一切，或慢节奏稳健长镜头）。
- **BGM/音效 (Audio Cue):** BGM 在该 Beat 的情绪（如：高潮、低沉、转折），以及是否有特殊的转场音效（如：Wwoosh 声、咔哒声）。

---

## 输出格式 (Output Format)
请严格按照以下 JSON 格式输出，不要包含任何解释性文本，确保可被后端系统直接解析：

```json
[
  {
    "beat_index": 1,
    "function": "Hook",
    "timestamp": {
      "start": "00:00:00.000",
      "end": "00:00:03.200",
      "duration_seconds": 3.2
    },
    "narration": "这里是该 Beat 的完整语音文案",
    "style_analysis": {
      "visual_type": "3D Render / Minimalist UI / Real-world footage",
      "composition": "Centered, close-up shot",
      "color_palette": ["#111111", "#00FFCC"],
      "background": "Dark gradient with subtle grid particles",
      "motion_camera": "Slow zoom-in",
      "motion_assets": "Floating elements with spring physics",
      "transition_out": "Hard cut on the beat",
      "pacing": "Fast-paced, high energy"
    },
    "typography": {
      "text_present": true,
      "font_summary": "Bold sans-serif, yellow color with black stroke",
      "position": "Center-bottom",
      "animation": "Pop-up word by word"
    }
  }
]
"""

In [18]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).parent.resolve()))

from openai import OpenAI

from src.video import VideoClip

video_path = Path("../tests/videos/抖音2026529-207530.mp4")
video = VideoClip(video_path)

client = OpenAI(
    api_key=os.getenv("API_KEY", ""),
    base_url=os.getenv("BASE_URL", ""),
)

try:
    completion = client.chat.completions.create(
        model=os.getenv("MODEL", ""),
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "video_url",
                        "video_url": {
                            "url": f"data:video/mp4;base64,{video.to_base64()}"
                        },
                    },
                    {"type": "text", "text": STRUCTURE_EXTRACTION_PROMPT},
                ],
            },  # type: ignore
        ],
        # 传输视频
        temperature=0.2,
    )

    # 打印返回结果
    print(completion.choices[0].message.content)

except Exception as e:
    print(f"调用失败，错误信息: {e}")

```STYLE.md
# Style Reference: Dark Typo Punch 暗黑文字暴击
## 1. Overview & Mood (视觉基调与世界观)
- **Concept / Metaphor**: 观众置身于完全空无一物的纯黑虚空之中，所有文字如同悬浮在暗空间里的自发光标牌，用极致的明暗反差逐个抛出价值命题，最终用“11亿>10亿”的荒诞数字反转完成对观众认知的精准重击，全程没有多余视觉元素，所有注意力完全被发光文字锚定。
- **Mood Direction**: 冷峻硬核/荒诞洗脑/高反差叙事/充满黑色幽默的网感冲击力
- **Energy Level**: High
```
